In [1]:
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

In [2]:
customer_message = """
Hi, I received my blue jacket yesterday and it arrived damaged.
I also think I was charged twice.
I want a full refund, not a replacement.
My order number is ORD-77819.
"""

In [3]:
tools = [
    {
        "name": "verify_customer",
        "description": "Verify the currently authenticated customer session.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "lookup_order",
        "description": "Look up an order for a verified customer.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"}
            },
            "required": ["customer_id", "order_id"]
        }
    },
    {
        "name": "check_return_policy",
        "description": "Check return and refund policy for a specific item.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"},
                "item_id": {"type": "string"},
                "reason": {"type": "string"}
            },
            "required": ["customer_id", "order_id", "item_id", "reason"]
        }
    },
    {
        "name": "investigate_duplicate_charge",
        "description": "Check whether an order has a duplicate captured charge.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"}
            },
            "required": ["customer_id", "order_id"]
        }
    },
    {
        "name": "process_refund",
        "description": "Process a refund if verification and policy gates allow it.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"},
                "refund_amount": {"type": "number"},
                "reason": {"type": "string"}
            },
            "required": ["customer_id", "order_id", "refund_amount", "reason"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Create a structured human escalation handoff.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "order_id": {"type": "string"},
                "root_cause": {"type": "string"},
                "refund_amount": {"type": "number"},
                "recommended_action": {"type": "string"},
                "evidence": {"type": "array", "items": {"type": "string"}},
                "missing_information": {"type": "array", "items": {"type": "string"}},
                "escalation_reason": {"type": "string"}
            },
            "required": [
                "customer_id",
                "order_id",
                "root_cause",
                "recommended_action",
                "escalation_reason"
            ]
        }
    },
    {
        "name": "extract_case_facts",
        "description": "Extract initial structured case facts from the customer message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_message": {"type": "string"}
            },
            "required": ["customer_message"]
        }
    }
]

In [4]:
SESSION = {
    "session_token": "valid_session"
}

def verify_customer():
    if SESSION["session_token"] != "valid_session":
        return {
            "status": "error",
            "error_code": "CUSTOMER_NOT_VERIFIED",
            "message": "Customer verification failed."
        }

    return {
        "status": "success",
        "customer_id": "CUS-1842"
    }

In [5]:
def lookup_order(customer_id: str, order_id: str):
    if customer_id != "CUS-1842" or order_id != "ORD-77819":
        return {
            "status": "error",
            "error_code": "ORDER_NOT_FOUND"
        }

    return {
        "status": "success",
        "order_id": "ORD-77819",
        "item_id": "jacket_blue_m",
        "item_name": "Blue Jacket",
        "price": 149.00,
        "delivered_at": "2026-06-24",
        "flags": ["damage_claim"]
    }

In [6]:
def check_return_policy(customer_id: str, order_id: str, item_id: str, reason: str):
    return {
        "status": "success",
        "policy_id": "returns-v7",
        "eligible": True,
        "automatic_refund_limit": 75.00,
    }

In [7]:
def investigate_duplicate_charge(customer_id: str, order_id: str):
    return {
        "status": "success",
        "duplicate_charge_found": False,
        "payment_status": "second authorization pending, not captured"
    }

In [8]:
def process_refund(customer_id: str, order_id: str, refund_amount: float, reason: str):
    if refund_amount > 75.00:
        return {
            "status": "blocked",
            "error_code": "REFUND_LIMIT_EXCEEDED",
            "message": "Refund amount exceeds automatic approval limit.",
            "next_step": "escalate_to_human"
        }

    return {
        "status": "success",
        "refund_id": "REF-9001",
        "refund_amount": refund_amount
    }

In [9]:
def extract_case_facts(customer_message: str):
    return {
        "status": "success",
        "order_id": "ORD-77819",
        "issues": ["damaged_item", "possible_duplicate_charge"],
        "customer_expectation": "full refund, not replacement"
    }

In [13]:
def run_tool(name: str, tool_input: dict):
    if name == "verify_customer":
        return verify_customer()

    if name == "lookup_order":
        return lookup_order(**tool_input)

    if name == "check_return_policy":
        return check_return_policy(**tool_input)

    if name == "investigate_duplicate_charge":
        return investigate_duplicate_charge(**tool_input)

    if name == "process_refund":
        return process_refund(**tool_input)

    if name == "escalate_to_human":
        return {
            "status": "success",
            "handoff_id": "HANDOFF-3007",
            **tool_input
        }
    
    if name == "extract_case_facts":
        return extract_case_facts(**tool_input)

    return {
        "status": "error",
        "error_code": "UNKNOWN_TOOL"
    }

In [11]:
def update_case_facts(case_facts: dict, tool_name: str, result: dict):
    if result.get("status") == "error":
        case_facts["missing_information"].append({
            "tool": tool_name,
            "error_code": result.get("error_code")
        })
        return

    if tool_name == "extract_case_facts":
        case_facts["order_id"] = result.get("order_id")
        case_facts["issues"] = result.get("issues", [])
        case_facts["customer_expectation"] = result.get("customer_expectation")

    if tool_name == "verify_customer":
        case_facts["customer_id"] = result.get("customer_id")

    if tool_name == "lookup_order":
        case_facts["order_id"] = result.get("order_id")
        case_facts["item_id"] = result.get("item_id")
        case_facts["refund_amount_requested"] = result.get("price")
        case_facts["evidence"].extend(result.get("flags", []))

    if tool_name == "check_return_policy":
        case_facts["policy"] = {
            "policy_id": result.get("policy_id"),
            "eligible": result.get("eligible"),
            "automatic_refund_limit": result.get("automatic_refund_limit"),
            "recommended_action": result.get("recommended_action")
        }

    if tool_name == "investigate_duplicate_charge":
        case_facts["payment"] = {
            "duplicate_charge_found": result.get("duplicate_charge_found"),
            "payment_status": result.get("payment_status")
        }

    if tool_name == "process_refund" and result.get("status") == "blocked":
        case_facts["escalation_reason"] = result.get("message")

    if tool_name == "escalate_to_human":
        case_facts["handoff_id"] = result.get("handoff_id")

In [15]:
case_facts = {
    "customer_id": None,
    "order_id": None,
    "item_id": None,
    "issues": [],
    "customer_expectation": None,
    "refund_amount_requested": None,
    "evidence": [],
    "missing_information": [],
    "policy": None,
    "payment": None,
    "escalation_reason": None
}

messages = [
    {
        "role": "user",
        "content": f"""
Customer message:
{customer_message}

Current case facts:
{json.dumps(case_facts, indent=2)}

Handle the support case.
First, use extract_case_facts to extract the initial order ID, issue types, and customer expectation.
Then verify the currently authenticated customer session.
After verification, inspect the order, check return policy, and investigate duplicate charge if needed.
If the customer requests a refund and the item is policy-eligible, request process_refund.
Do not decide by yourself whether the refund is allowed.
The backend refund tool will enforce refund limits and return success or a structured blocked result.
If process_refund returns a blocked result, use escalate_to_human with a structured handoff.
Do not produce the final customer response until either:
- the refund has been processed successfully, or
- the refund was blocked and the case was escalated to a human.
"""
    }
]

while True:
    message = client.messages.create(
        model=model,
        max_tokens=1200,
        tools=tools,
        messages=messages
    )

    messages.append({
        "role": "assistant",
        "content": message.content
    })

    if message.stop_reason == "tool_use":
        tool_results = []

        for block in message.content:
            if block.type != "tool_use":
                continue

            result = run_tool(block.name, block.input)

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": json.dumps(result)
            })

            update_case_facts(case_facts, block.name, result)

            print("Claude requested:", block.name)

        messages.append({
            "role": "user",
            "content": tool_results
        })

        continue

    if message.stop_reason == "end_turn":
        print(message.content[0].text)
        break

    handoff = run_tool("escalate_to_human", {
        "case_facts": case_facts,
        "escalation_reason": f"Unexpected stop reason: {message.stop_reason}"
    })

    print(handoff)
    break

Claude requested: extract_case_facts
Claude requested: verify_customer
Claude requested: lookup_order
Claude requested: check_return_policy
Claude requested: investigate_duplicate_charge
Claude requested: process_refund
Claude requested: escalate_to_human
The case has been fully escalated. Here's the summary for the customer:

---

## Here's Where Things Stand, [Customer]

Thank you for reaching out — I'm sorry to hear your Blue Jacket arrived damaged. I've reviewed your case thoroughly and here's what I found:

### ✅ What We Confirmed
| Item | Detail |
|---|---|
| **Order** | ORD-77819 — Blue Jacket, delivered June 24 |
| **Damage Claim** | Logged and flagged on your order ✔️ |
| **Refund Eligibility** | Your item **is eligible** for a return/refund |
| **Suspected Duplicate Charge** | No second charge was actually captured — there is a **pending authorization** on your account, but it has **not been billed**. It should drop off automatically, but our team will monitor it. |

### ⏳ Wh